# Haven W1 Phase 1: the stock Qwen3.5-4B as its own student (rejection sampling, no training)

**Do not run until both gates on #123 are met:** Astra's **DECISION:** sign-off on `labs/weights/PLAN.md`, and the M4 reserve digest posted on #119. Phase 0 built this notebook and did not run it.

What it does (PLAN.md, W1 Stage A, steps 1-4): about 200 training episodes from fresh secret seeds, 8 attempts each at temperature 0.7, inside the full HARE harness (advisory checks, gateway, verifier, one correction). An attempt is kept only if the adjudicator passes it against gold with zero blocked or duplicate calls, at most 2 per episode. Episodes with no kept attempt go to W2. Gate G1: 15-85% of attempts pass.

**Before running:** attach the private dataset holding `w1-pilot.episodes.jsonl` (made by `gen_hare.py generate`) and set `EPISODES_SHA256` below to the value posted on #123. Accelerator **GPU T4 x2**, Internet **On**. Keep the notebook private: `attempts.jsonl` and `kept-rows.jsonl` hold full synthetic transcripts and training gold. Only `summary.json` leaves the session.

Estimated GPU time: about 3 hours (PLAN.md); the run replaces this with a measured number.

In [ ]:
EPISODES_SHA256 = ""        # REQUIRED: the w1-pilot episodes sha256 posted on #123
EPISODES_NAME = "w1-pilot.episodes.jsonl"
ATTEMPTS = 8
TEMPERATURE = 0.7
MAX_TOKENS = 1024
DEADLINE_MINUTES = 600
MAIN_COMMIT = "53387e74a0ad90786ac3ea16753c8ad99142e9df"
MODEL = {"repo": "unsloth/Qwen3.5-4B-GGUF", "revision": "e87f176479d0855a907a41277aca2f8ee7a09523", "file": "Qwen3.5-4B-Q4_K_M.gguf", "sha256": "00fe7986ff5f6b463e62455821146049db6f9313603938a70800d1fb69ef11a4"}
WORK = "/kaggle/working/w1-phase1"
MODELS = "/tmp/haven-models"
assert len(EPISODES_SHA256) == 64, "set EPISODES_SHA256 to the value posted on #123"


In [ ]:
import subprocess, os, sys, shutil, json, hashlib, base64, io, tarfile, pathlib, time, glob
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout or "")[-3000:], (r.stderr or "")[-3000:], flush=True)
    if check and r.returncode: raise SystemExit(f"failed: {cmd}")
    return r.stdout
def stream(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout: print(line, end="", flush=True)
    assert p.wait() == 0, f"failed: {cmd}"
sh("nvidia-smi --query-gpu=name,memory.total --format=csv"); sh("nproc; free -g | head -2")
os.makedirs(WORK, exist_ok=True); os.makedirs(MODELS, exist_ok=True)


In [ ]:
if not os.path.isdir("/tmp/haven/.git"):
    sh("git clone -q --filter=blob:none https://github.com/erpzz/haven /tmp/haven")
sh(f"git -C /tmp/haven checkout -q {MAIN_COMMIT} && git -C /tmp/haven log -1 --format='%h %s'")
BLOB = "..."
raw = base64.b64decode(BLOB)
assert hashlib.sha256(raw).hexdigest() == "c98990eb1df7d9a656233410d5bd2d350867326f1d78e714fab9a5a90c555698", "embedded Phase 0 bundle corrupted"
dest = pathlib.Path("/tmp/haven/labs/weights")
dest.mkdir(parents=True, exist_ok=True)
shutil.rmtree(dest / "phase0", ignore_errors=True)
tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz").extractall(dest, filter="data")
P0 = dest / "phase0"
sh(f"cd {P0} && python3 fetch_deps.py --verify")
sh(f"cd {P0} && python3 -m unittest discover -s tests 2>&1 | tail -4")


In [ ]:
# Kaggle llama.cpp b11146 CUDA build helpers, copied verbatim from PR #115
# (labs/intelligence-bakeoff/system0/kaggle/build_notebook.py at 7f5af0d), where they built the engine
# for Kaggle run 1. Kept as a separate file so a diff against the source is trivial.
STUB_DEB = ("https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/cuda-driver-dev-13-0_13.0.88-1_amd64.deb",
            "3ffcc3fc69aa914629a97839aaf2b624508fbae9d87f187231795745a6be2550")
def find_libcuda():
    # every libcuda.so* the loader cache, nvidia-smi's own libraries or common driver dirs know about
    sh("echo LD_LIBRARY_PATH=$LD_LIBRARY_PATH; ldd $(command -v nvidia-smi) 2>/dev/null | grep -i nvidia", check=False)
    out = sh("ldconfig -p | grep -o '/[^ ]*libcuda[.]so[.0-9]*' ; "
             "for d in $(ldd $(command -v nvidia-smi) 2>/dev/null | grep -o '/[^ ]*libnvidia-ml[^ ]*' | xargs -r -n1 dirname); do ls $d/libcuda.so* 2>/dev/null; done; "
             "find /usr/local/nvidia /usr/lib64-nvidia /usr/local/cuda /usr/local/cuda-13.0 /usr/lib /usr/lib64 /opt "
             "-name 'libcuda.so*' 2>/dev/null | head -40", check=False)
    found = {p.strip() for p in out.split() if os.path.basename(p.strip()).startswith("libcuda.so") and os.path.exists(p.strip())}
    if not [p for p in found if os.path.getsize(os.path.realpath(p)) > 5_000_000]:
        out = sh("timeout 180 find / -path /proc -prune -o -path /sys -prune -o -path /kaggle/input -prune -o "
                 "-name 'libcuda.so*' -print 2>/dev/null | head -20", check=False)
        found |= {p.strip() for p in out.split() if os.path.basename(p.strip()).startswith("libcuda.so") and os.path.exists(p.strip())}
    return sorted(found)
def is_real_driver(p):
    # the driver library is tens of MB; NVIDIA's link stub (and our renamed copies of it) is tiny.
    # cuda-*/compat holds a forward-compatibility libcuda for OLDER host drivers: loading it on a newer
    # host driver fails with "unsupported display driver / cuda driver combination", so never use it.
    return os.path.getsize(os.path.realpath(p)) > 5_000_000 and "/compat/" not in p
def host_drivers(found):
    real = [p for p in found if is_real_driver(p)]
    return sorted(real, key=lambda p: (not p.startswith("/usr/local/nvidia"), p))
def find_libcuda_runtime():
    real = host_drivers(find_libcuda())
    if real:  # host driver directory first, ahead of anything else on the path
        os.environ["LD_LIBRARY_PATH"] = os.path.dirname(real[0]) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
def libcuda_for_link():
    found = find_libcuda()
    print("libcuda candidates:", found)
    real = host_drivers(found)
    if real:  # make sure the server loads the host driver at run time, not a compat copy
        d = os.path.dirname(real[0])
        os.environ["LD_LIBRARY_PATH"] = d + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    stubs = [p for p in found if os.path.getsize(os.path.realpath(p)) <= 5_000_000 and "/libcuda-link" not in p]
    if stubs:
        stub = stubs[0]
    else:
        deb = "/tmp/cuda-driver-dev.deb"
        sh(f"curl -sSL --retry 5 -o {deb} '{STUB_DEB[0]}'")
        assert hashlib.sha256(open(deb, "rb").read()).hexdigest() == STUB_DEB[1], "stub package hash mismatch"
        sh(f"rm -rf /tmp/cuda-stub && dpkg-deb -x {deb} /tmp/cuda-stub")
        stub = sh("find /tmp/cuda-stub -name libcuda.so | head -1").strip()
        assert stub, "no libcuda.so in the stub package"
    # the linker resolves the dependency by its soname (libcuda.so.1); give it both names in one writable dir
    sh(f"rm -rf /tmp/libcuda-link && mkdir -p /tmp/libcuda-link && cp {stub} /tmp/libcuda-link/libcuda.so "
       "&& ln -s libcuda.so /tmp/libcuda-link/libcuda.so.1")
    return "/tmp/libcuda-link/libcuda.so"
def cached_build(tag):
    # a private Kaggle dataset holding a prebuilt llama-server (+ libs) and a sha256 manifest
    import glob
    for man in glob.glob(f"/kaggle/input/**/haven-llama-{tag}-cuda13-sm75/manifest.json", recursive=True):
        root = os.path.dirname(man)
        files = json.load(open(man))["files"]
        if all(hashlib.sha256(open(os.path.join(root, f), "rb").read()).hexdigest() == h for f, h in files.items()):
            return root
        print("cached build failed its hash check, ignoring:", root)
    return None
def build_llama(tag, prefix):
    engine = f"{prefix}/build/bin/llama-server"
    cache = cached_build(tag) if not os.path.exists(engine) else None
    if cache:
        sh(f"mkdir -p {prefix}/build/bin && cp -a {cache}/bin/. {prefix}/build/bin/ && chmod +x {engine}")
        print("using cached build from", cache)
        os.environ["LD_LIBRARY_PATH"] = (f"{prefix}/build/bin:/usr/local/cuda/lib64:/usr/local/cuda/targets/x86_64-linux/lib:"
                                         + os.environ.get("LD_LIBRARY_PATH", ""))
        find_libcuda_runtime()
    if not os.path.exists(engine):
        sh(f"git clone -q --depth 1 --branch {tag} https://github.com/ggml-org/llama.cpp {prefix}")
        t0 = time.time()
        drv_lib = libcuda_for_link()
        link = f"-Wl,-rpath-link,{os.path.dirname(drv_lib)}"
        flags = (f"-DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DCUDA_cuda_driver_LIBRARY={drv_lib} "
                 f"'-DCMAKE_EXE_LINKER_FLAGS={link}' '-DCMAKE_SHARED_LINKER_FLAGS={link}' -DLLAMA_CURL=OFF "
                 "-DLLAMA_USE_PREBUILT_UI=OFF -DLLAMA_BUILD_UI=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF")
        cmake = "cmake"
        sh(f"cd {prefix} && {cmake} -S . -B build {flags} > /tmp/cmake.log 2>&1 || (tail -40 /tmp/cmake.log; false)")
        sh(f"cd {prefix} && {cmake} --build build --target llama-server -j4 > /tmp/build.log 2>&1 || (tail -60 /tmp/build.log; false)")
        print(f"{tag} build minutes: {(time.time()-t0)/60:.1f}")
    sh("nvidia-smi --query-gpu=driver_version --format=csv,noheader | head -1", check=False)
    sh(f"ldd {os.path.dirname(engine)}/libggml-cuda.so | grep -E 'libcuda|libcudart|libcublas'", check=False)
    out = sh(f"{engine} --list-devices 2>&1", check=False)
    import re as _re
    if "failed to initialize CUDA" in out or not _re.search(r"^\s*CUDA\d+:", out, _re.M):
        raise SystemExit("llama-server cannot use the GPUs (see the libcuda lines above); stopping instead of running on CPU")
    return engine

ENGINE = build_llama("b11146", "/tmp/llama.cpp")   # the run-1 engine


In [ ]:
def fetch(spec):
    path = f"{MODELS}/{spec['file']}"
    if not os.path.exists(path):
        url = f"https://huggingface.co/{spec['repo']}/resolve/{spec['revision']}/{spec['file']}"
        sh(f"curl -sSL --retry 5 -o {path}.part '{url}' && mv {path}.part {path}")
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(1 << 24), b""): h.update(c)
    assert h.hexdigest() == spec["sha256"], f"sha256 mismatch for {spec['file']}"
    return path
M4 = fetch(MODEL)
found = glob.glob(f"/kaggle/input/**/{EPISODES_NAME}", recursive=True)
assert len(found) == 1, f"attach exactly one private dataset holding {EPISODES_NAME} (found {found})"
EPISODES = found[0]
gpus = sh("nvidia-smi --query-gpu=name --format=csv,noheader").strip().splitlines()
GPU_IDS = ",".join(str(i) for i in range(len(gpus)))
HW = f"Kaggle {len(gpus)}x {gpus[0]}; one llama-server per GPU, identical args"


In [ ]:
stream(f"cd {P0} && python3 kaggle/w1_phase1.py --engine {ENGINE} --model {M4} --episodes {EPISODES} "
       f"--episodes-sha256 {EPISODES_SHA256} --out {WORK} --gpus {GPU_IDS} --attempts {ATTEMPTS} "
       f"--temperature {TEMPERATURE} --max-tokens {MAX_TOKENS} --deadline-minutes {DEADLINE_MINUTES} --hardware '{HW}'")


In [ ]:
# Only the summary leaves the session. Kept rows and transcripts stay private.
shutil.copy(f"{WORK}/summary.json", "/kaggle/working/w1-phase1-summary.json")
print(json.dumps(json.load(open("/kaggle/working/w1-phase1-summary.json"))["summary"], indent=2))
